In [4]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib.ticker import PercentFormatter


# ---------------------------------------------------------------------
# Localizar TFM_resp_dis
# ---------------------------------------------------------------------
current_dir = Path.cwd().resolve()

possible_roots = [
    current_dir,
    current_dir.parent,
    current_dir / "TFM_resp_dis",
    current_dir.parent / "TFM_resp_dis",
]

project_root = next(
    (
        path
        for path in possible_roots
        if (
            path
            / "train_stage2_dry_wet_wavelet_scattering_recording_linear.py"
        ).exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "No se ha encontrado el repositorio TFM_resp_dis."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# ---------------------------------------------------------------------
# Cargar exactamente las features del experimento WST recording + LR
# ---------------------------------------------------------------------
import train_stage2_dry_wet_wavelet_scattering_recording_linear as wst_lr

preset = "paper_q8_q1_t500_full"

event_data, extraction_configuration = (
    wst_lr.FEATURE_LOADER(preset)
)

recording_data = wst_lr.build_recording_data(event_data)

X_train = np.asarray(
    recording_data.x_train,
    dtype=np.float64,
)

print("Forma de TRAIN antes de PCA:", X_train.shape)

if X_train.shape[1] != 1932:
    raise ValueError(
        f"Se esperaban 1932 variables, pero se encontraron "
        f"{X_train.shape[1]}."
    )


# ---------------------------------------------------------------------
# StandardScaler: igual que en el pipeline original
# ---------------------------------------------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)


# ---------------------------------------------------------------------
# Calcular los primeros 644 componentes
# ---------------------------------------------------------------------
n_components_to_plot = min(
    1932,
    X_train_scaled.shape[0] - 1,
    X_train_scaled.shape[1],
)

pca_full_curve = PCA(
    n_components=n_components_to_plot,
    svd_solver="randomized",
    random_state=42,
)

pca_full_curve.fit(X_train_scaled)

explained_variance = (
    pca_full_curve.explained_variance_ratio_
)

cumulative_variance = np.cumsum(
    explained_variance
)

components = np.arange(
    1,
    len(cumulative_variance) + 1,
)

variance_at_128 = cumulative_variance[127]

print(
    "Varianza acumulada con 128 componentes: "
    f"{variance_at_128:.4f} "
    f"({variance_at_128 * 100:.2f} %)"
)


# ---------------------------------------------------------------------
# Guardar los datos de la curva
# ---------------------------------------------------------------------
figures_dir = project_root / "figures_tfm_stage2"
figures_dir.mkdir(parents=True, exist_ok=True)

pca_curve_data = pd.DataFrame(
    {
        "n_components": components,
        "explained_variance_ratio": explained_variance,
        "cumulative_explained_variance_ratio": cumulative_variance,
    }
)

pca_curve_data.to_csv(
    figures_dir
    / "stage2_wst_pca_cumulative_explained_variance.csv",
    index=False,
)


# ---------------------------------------------------------------------
# Gráfica
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.0, 4.8))

ax.plot(
    components,
    cumulative_variance,
    color="#2F6FA5",
    linewidth=2,
)

ax.scatter(
    128,
    variance_at_128,
    color="#C94C4C",
    s=42,
    zorder=4,
)

ax.axvline(
    128,
    color="#C94C4C",
    linestyle=":",
    linewidth=1,
    alpha=0.45,
)

ax.axhline(
    variance_at_128,
    color="#C94C4C",
    linestyle=":",
    linewidth=1,
    alpha=0.45,
)

ax.annotate(
    f"128 components\n"
    f"{variance_at_128 * 100:.1f}% variance",
    xy=(128, variance_at_128),
    xytext=(165, variance_at_128 - 0.055),
    arrowprops={
        "arrowstyle": "->",
        "color": "#555555",
        "linewidth": 0.9,
    },
    fontsize=9,
    ha="left",
)

ax.set_xlim(1, n_components_to_plot)

# Escala ampliada, igual que en la figura de referencia
ax.set_ylim(0.50, 1.005)

ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1.0)
)

ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title(
    "PCA cumulative explained variance — Stage 2 WST"
)

ax.grid(
    linestyle="--",
    linewidth=0.7,
    alpha=0.30,
)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()

fig.savefig(
    figures_dir
    / "stage2_wst_pca_cumulative_explained_variance.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    figures_dir
    / "stage2_wst_pca_cumulative_explained_variance.pdf",
    bbox_inches="tight",
)

plt.show()

Forma de TRAIN antes de PCA: (1623, 1932)
Varianza acumulada con 128 componentes: 0.9522 (95.22 %)


C:\Users\Usuario\AppData\Local\Temp\ipykernel_45332\867005308.py:230: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
